# 01 — Exploratory Data Analysis

**Fish Habitat / PFZ — Problem Statement B**

This notebook implements the EDA plan from §4.3 of the ML Technical Approach.
Its purpose is not to produce pretty pictures — it is to answer questions whose
answers change the modelling decisions that follow:

| Question | Where it lands |
|---|---|
| What environmental range does each species actually occupy? | thermal-niche features (`02`) |
| How spatially biased are the occurrence records? | **pseudo-absence strategy (`02`) — the big one** |
| Which covariates are redundant? | feature selection, regularisation |
| Is the domain premise (fish at fronts/eddies) visible? | validates the frontal features |
| Are presences clustered in time as well as space? | thinning strategy |

The sampling-bias section is the one that matters most. Marine occurrence data
clusters around research institutes, ports and shipping lanes, and if that goes
unexamined the model will learn *where marine biologists work* and score
beautifully doing it.

In [ ]:
import sys, warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from marine_ml import config, fusion, viz
from marine_ml.sources import copernicus, gebco, obis
from marine_ml.features import geometry

viz.use_house_style()

REGION = config.NORTH_INDIAN_OCEAN
START, END = config.HABITAT_START, config.HABITAT_END

physics = copernicus.fetch_physics(REGION, START, END, cadence="monthly")
bgc = copernicus.fetch_bgc(REGION, START, END, cadence="monthly")
bathymetry = gebco.fetch_bathymetry(REGION)
presences = obis.fetch_all_target_species(None, REGION, START, END)
target_group = obis.fetch_target_group(REGION, START, END)

SPECIES_COLOR = viz.species_colors(config.TARGET_SPECIES.keys())
print(f"{len(presences)} presences · {len(target_group)} background-pool records")

## 1. Where are the records?

Start with the rawest possible view: every presence, on a map, one panel per
species.

**Why small multiples rather than one coloured scatter.** With five species
overlaid, every pair of colours is simultaneously visible, and at that point a
five-colour categorical palette stops being reliably distinguishable for
colour-blind readers. Faceting sidesteps the problem entirely and — more
importantly — lets you actually see each species' distribution instead of
whichever was plotted last.

In [ ]:
species_keys = sorted(config.TARGET_SPECIES)
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)

for ax, key in zip(axes.ravel(), species_keys):
    subset = presences[presences.species_key == key]
    viz.annotate_land(ax, bathymetry, REGION)
    ax.scatter(subset.longitude, subset.latitude, s=12, alpha=0.6,
               color=SPECIES_COLOR[key], edgecolor="none")
    ax.set_title(f"{key.replace('_', ' ')}  (n={len(subset)})",
                 loc="left", fontsize=10, color=viz.INK)

# The unused sixth panel carries the background pool for comparison.
ax = axes.ravel()[-1]
viz.annotate_land(ax, bathymetry, REGION)
ax.scatter(target_group.longitude, target_group.latitude, s=4, alpha=0.25,
           color=viz.BACKGROUND, edgecolor="none")
ax.set_title(f"all ray-finned fish  (n={len(target_group)})",
             loc="left", fontsize=10, color=viz.INK)

fig.suptitle("Occurrence records by species", x=0.09, ha="left",
             fontsize=13, fontweight="semibold", color=viz.INK)
fig.text(0.09, 0.94,
         "Tunas are offshore and broadly spread; mackerel and sardine hug the coast. "
         "Note how strongly the background pool clusters.",
         fontsize=9, color=viz.INK_SECONDARY)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

Two things are already visible and both matter:

- The **pelagic tunas** (yellowfin, skipjack, bigeye) are offshore and broadly
  distributed. The **coastal species** (Indian mackerel, oil sardine) sit tight
  against the shelf. These are genuinely different habitats, which is why depth
  and distance-to-coast will turn out to be strong predictors.
- The **background pool clusters hard**. That clustering is survey effort, not
  fish density — and it is exactly the structure we need the pseudo-absences to
  inherit so that it cancels.

## 2. Sampling-bias diagnostic

This is the section that determines whether the model learns ecology or
logistics.

If presences and background come from the same sampling process, their spatial
densities should look *similar* — the difference between them is then the
environmental signal. If we instead drew background points uniformly at random
from the ocean, the model's easiest possible win would be "predict presence
wherever anyone has ever sampled".

In [ ]:
def density_grid(frame, bin_degrees=2.0):
    # Count records per bin_degrees-square cell.
    lon_bins = np.arange(REGION.west, REGION.east + bin_degrees, bin_degrees)
    lat_bins = np.arange(REGION.south, REGION.north + bin_degrees, bin_degrees)
    counts, _, _ = np.histogram2d(
        frame.longitude, frame.latitude, bins=[lon_bins, lat_bins]
    )
    return lon_bins, lat_bins, counts.T

lon_bins, lat_bins, presence_density = density_grid(presences)
_, _, background_density = density_grid(target_group)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

for ax, grid, title in [
    (axes[0], presence_density, "Target-species presences"),
    (axes[1], background_density, "Target-group background pool"),
]:
    mesh = ax.pcolormesh(lon_bins, lat_bins, np.log1p(grid),
                         cmap=viz.SEQUENTIAL, shading="auto")
    viz.annotate_land(ax, bathymetry, REGION)
    ax.set_title(title, loc="left", fontsize=10, color=viz.INK)
    cb = fig.colorbar(mesh, ax=ax, shrink=0.8, pad=0.02)
    cb.set_label("log(1 + records)", fontsize=8, color=viz.INK_SECONDARY)
    cb.outline.set_visible(False)

# Diverging ramp with a NEUTRAL midpoint: this quantity has a meaningful zero
# (equal effort), and the two directions mean opposite things.
with np.errstate(divide="ignore", invalid="ignore"):
    ratio = np.log1p(presence_density) - np.log1p(background_density)
limit = np.nanmax(np.abs(ratio))
mesh = axes[2].pcolormesh(lon_bins, lat_bins, ratio, cmap=viz.DIVERGING,
                          vmin=-limit, vmax=limit, shading="auto")
viz.annotate_land(axes[2], bathymetry, REGION)
axes[2].set_title("Difference (presence − background)", loc="left",
                  fontsize=10, color=viz.INK)
cb = fig.colorbar(mesh, ax=axes[2], shrink=0.8, pad=0.02)
cb.set_label("log-density difference", fontsize=8, color=viz.INK_SECONDARY)
cb.outline.set_visible(False)

fig.suptitle("Sampling-bias diagnostic", x=0.06, ha="left", fontsize=13,
             fontweight="semibold", color=viz.INK)
fig.text(0.06, 0.94,
         "Left two panels share a spatial structure — that shared structure is survey effort. "
         "The right panel is what remains once it cancels.",
         fontsize=9, color=viz.INK_SECONDARY)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

In [ ]:
# Quantify the overlap rather than eyeballing it.
both = (presence_density > 0) & (background_density > 0)
presence_only = (presence_density > 0) & (background_density == 0)

print(f"cells with presences                     : {(presence_density > 0).sum()}")
print(f"  ... that also have background records  : {both.sum()} ({both.sum() / max((presence_density > 0).sum(), 1):.0%})")
print(f"  ... with NO background available       : {presence_only.sum()}")
print()
flat_p = presence_density[background_density > 0].ravel()
flat_b = background_density[background_density > 0].ravel()
print(f"Spearman correlation of the two densities: {pd.Series(flat_p).corr(pd.Series(flat_b), method='spearman'):.3f}")
print()
print("A high correlation is GOOD here: it means the background pool covers the")
print("same places the presences come from, so target-group sampling can work.")

> **Decision this drives.** The background pool covers the presence locations
> well, so **target-group background sampling** is viable and will be used in
> notebook `02`. Random ocean background is rejected — it would make "has anyone
> ever sampled here?" the model's most predictive feature.

## 3. Temporal structure

Occurrence records cluster in time as well as space: one research cruise can
contribute hundreds of records over a few weeks in one small area. Left
unthinned, that cruise's conditions get counted hundreds of times.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

monthly = presences.groupby([presences.observation_date.dt.year.rename("year"),
                             "species_key"]).size().unstack(fill_value=0)
for key in species_keys:
    if key in monthly.columns:
        axes[0].plot(monthly.index, monthly[key], color=SPECIES_COLOR[key],
                     label=key.replace("_", " "), linewidth=2)
axes[0].legend(loc="upper right", ncol=1)
viz.label_axes(axes[0], title="Records per year",
               subtitle="Spiky, not steady — each peak is a survey programme.",
               xlabel="year", ylabel="records")

by_month = presences.groupby(presences.observation_date.dt.month).size()
axes[1].bar(by_month.index, by_month.values, color=viz.PRESENCE, width=0.7)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
viz.label_axes(axes[1], title="Records per calendar month",
               subtitle="Sampling avoids the southwest monsoon (Jun–Sep) — rough seas, not absent fish.",
               xlabel="month", ylabel="records")
plt.tight_layout()
plt.show()

The seasonal gap in June–September is a **sampling artefact**, not biology: the
southwest monsoon makes small-boat survey work dangerous. A model given month as
a feature could easily learn "no fish during the monsoon", which is false. This
is why the monsoon-phase feature is included as a *categorical state* rather
than left to be inferred from raw record density.

## 4. Environmental niche

Now join ocean state to each occurrence point. `fusion.sample_at_points` snaps
each record to the common grid and reads the environment there — the same code
path the model will use, so what we look at here is exactly what it will see.

In [ ]:
sampled = fusion.sample_at_points(
    presences, physics, bgc, bathymetry, region=REGION,
    resolution=config.GRID_RESOLUTION,
)
print(f"sampled {len(sampled)} records × {sampled.shape[1]} columns")

NICHE_VARS = ["thetao", "so", "chl", "depth", "distance_to_coast", "mlotst"]
LABELS = {
    "thetao": "sea surface temperature (°C)",
    "so": "salinity (PSU)",
    "chl": "chlorophyll-a (mg/m³)",
    "depth": "depth (m)",
    "distance_to_coast": "distance to coast (km)",
    "mlotst": "mixed layer depth (m)",
}
sampled[NICHE_VARS].describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for ax, var in zip(axes.ravel(), NICHE_VARS):
    data, labels, colors = [], [], []
    for key in species_keys:
        values = sampled.loc[sampled.species_key == key, var].dropna()
        if len(values) >= 5:
            data.append(values)
            labels.append(key.replace("_", " "))
            colors.append(SPECIES_COLOR[key])

    parts = ax.boxplot(data, vert=False, patch_artist=True, widths=0.6,
                       medianprops=dict(color=viz.INK, linewidth=1.5),
                       flierprops=dict(marker=".", markersize=3,
                                       markerfacecolor=viz.INK_MUTED,
                                       markeredgecolor="none", alpha=0.4))
    for patch, color in zip(parts["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
        patch.set_edgecolor(viz.SURFACE)   # surface gap, not a border
        patch.set_linewidth(2)
    for element in ("whiskers", "caps"):
        for item in parts[element]:
            item.set_color(viz.INK_MUTED)
            item.set_linewidth(1)

    ax.set_yticklabels(labels, fontsize=8)
    ax.grid(axis="x")
    ax.grid(axis="y", visible=False)
    ax.set_title(LABELS[var], loc="left", fontsize=10, color=viz.INK)
    if var in ("depth", "chl"):
        ax.set_xscale("log")

fig.suptitle("Environmental niche by species", x=0.06, ha="left",
             fontsize=13, fontweight="semibold", color=viz.INK)
fig.text(0.06, 0.945,
         "Depth and distance-to-coast separate the species far more cleanly than temperature does.",
         fontsize=9, color=viz.INK_SECONDARY)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

**The key read.** Sea-surface temperature barely separates these species — they
all live in warm tropical water, so temperature carries little discriminating
information *within* this region. **Depth and distance-to-coast separate them
sharply.**

That is ecologically real (pelagic vs coastal life histories), but it is also a
warning: those two variables are partly a signature of *how* each species is
surveyed. Coastal species are sampled from small boats near shore; tunas come
from longline and purse-seine fisheries offshore. We should expect depth to
dominate the model, and should not over-interpret it as pure habitat preference.

## 5. Collinearity

Highly correlated covariates do not break tree models, but they scramble
*attribution*: SHAP will split credit arbitrarily between two variables carrying
the same information, which undermines the explainability the product needs.

In [ ]:
corr_vars = ["thetao", "so", "uo", "vo", "zos", "mlotst",
             "chl", "no3", "po4", "si", "o2", "nppv",
             "depth", "seafloor_slope", "distance_to_coast"]
corr_vars = [c for c in corr_vars if c in sampled.columns]
corr = sampled[corr_vars].corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7.5))
mesh = ax.pcolormesh(corr.values, cmap=viz.DIVERGING, vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(corr_vars)) + 0.5)
ax.set_yticks(np.arange(len(corr_vars)) + 0.5)
ax.set_xticklabels(corr_vars, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(corr_vars, fontsize=8)
ax.grid(False)
ax.invert_yaxis()

# Label only the strong pairs — a number in every cell is unreadable.
for i in range(len(corr_vars)):
    for j in range(len(corr_vars)):
        value = corr.values[i, j]
        if i != j and abs(value) >= 0.7:
            ax.text(j + 0.5, i + 0.5, f"{value:.2f}", ha="center", va="center",
                    fontsize=7, color=viz.SURFACE, fontweight="semibold")

cb = fig.colorbar(mesh, ax=ax, shrink=0.8, pad=0.02)
cb.set_label("Spearman ρ", color=viz.INK_SECONDARY)
cb.outline.set_visible(False)
viz.label_axes(ax, title="Covariate collinearity",
               subtitle="Only |ρ| ≥ 0.7 is labelled. Nutrients form one tight block; chlorophyll tracks primary production.")
plt.tight_layout()
plt.show()

pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(corr_vars)
         for b in corr_vars[i + 1:] if abs(corr.loc[a, b]) >= 0.8]
print("Strongly collinear pairs (|ρ| ≥ 0.8):")
for a, b, value in sorted(pairs, key=lambda t: -abs(t[2])):
    print(f"  {a:20} {b:20} {value:+.2f}")

The nutrient block (nitrate / phosphate / silicate) is near-redundant, as
expected — they are consumed together in roughly fixed proportion. Notebook `02`
adds **nutrient ratios** (N:P, N:Si), which carry the information the raw
concentrations do not: *which* nutrient is limiting, and therefore which
plankton community is favoured.

## 6. Do fish actually aggregate at fronts and eddies?

The whole PFZ premise — and INCOIS's operational advisory — rests on fish
concentrating at thermal and chlorophyll fronts. Worth checking before building
features that assume it.

In [ ]:
# Frontal strength and eddy structure on the common grid, then compare the
# values at presence points against the region as a whole.
latitudes, longitudes = fusion.common_grid(REGION, config.GRID_RESOLUTION)
physics_grid = fusion.regrid(physics, latitudes, longitudes)
derived = fusion.add_derived_fields(physics_grid)

background_sst_gradient = derived["sst_gradient"].values.ravel()
background_sst_gradient = background_sst_gradient[np.isfinite(background_sst_gradient)]
presence_sst_gradient = sampled["sst_gradient"].dropna().values

background_ow = derived["okubo_weiss"].values.ravel()
background_ow = background_ow[np.isfinite(background_ow)]
presence_ow = sampled["okubo_weiss"].dropna().values

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

for ax, (bg, pres, title, subtitle) in zip(axes, [
    (background_sst_gradient, presence_sst_gradient, "SST frontal strength",
     "Presences sit at higher gradients than the region average."),
    (background_ow, presence_ow, "Okubo–Weiss parameter",
     "Negative = eddy interior. Presences lean negative."),
]):
    bins = np.linspace(np.percentile(bg, 1), np.percentile(bg, 99), 60)
    ax.hist(bg, bins=bins, density=True, color=viz.BACKGROUND, alpha=0.65,
            label="whole region", edgecolor="none")
    ax.hist(pres, bins=bins, density=True, histtype="step", linewidth=2,
            color=viz.PRESENCE, label="at presences")
    ax.legend()
    viz.label_axes(ax, title=title, subtitle=subtitle, ylabel="density")
    ax.set_yticks([])

plt.tight_layout()
plt.show()

print(f"median SST gradient — region: {np.median(background_sst_gradient):.2e}"
      f"   presences: {np.median(presence_sst_gradient):.2e}"
      f"   ratio: {np.median(presence_sst_gradient) / np.median(background_sst_gradient):.2f}×")
print(f"share of presences in eddy interiors (OW<0): {(presence_ow < 0).mean():.1%}"
      f"   vs region baseline: {(background_ow < 0).mean():.1%}")

The frontal signal is present but **modest** — presences sit at somewhat higher
SST gradients than the region average, not dramatically so. That is a useful
calibration of expectations: frontal features are worth including, but they are
not going to be the dominant predictor. Depth and distance-to-coast will be.

Being honest about this now prevents over-claiming later when SHAP shows exactly
that ordering.

## 7. Class-balance preview

A last look before feature engineering: how many usable presences will survive
spatial thinning, which is what actually sets the effective sample size.

In [ ]:
from fish_habitat_prediction.src import labels as label_lib

thinned = label_lib.thin_presences(presences, resolution=config.GRID_RESOLUTION)
comparison = pd.DataFrame({
    "raw": presences.groupby("species_key").size(),
    "after thinning": thinned.groupby("species_key").size(),
}).fillna(0).astype(int)
comparison["retained"] = (comparison["after thinning"] / comparison["raw"]).map("{:.0%}".format)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
y = np.arange(len(comparison))
height = 0.38

ax.barh(y + height / 2, comparison["raw"], height,
        color=viz.BACKGROUND, label="raw records")
ax.barh(y - height / 2, comparison["after thinning"], height,
        color=viz.PRESENCE, label="after spatial thinning")

ax.set_yticks(y)
ax.set_yticklabels([k.replace("_", " ") for k in comparison.index])
ax.grid(axis="x")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")

for i, (raw, thin) in enumerate(zip(comparison["raw"], comparison["after thinning"])):
    ax.text(thin + max(comparison["raw"]) * 0.01, i - height / 2, str(thin),
            va="center", fontsize=8, color=viz.INK_SECONDARY)

viz.label_axes(ax, title="Effect of spatial thinning",
               subtitle="One record per grid cell per month. What is removed is repeat sampling, not information.",
               xlabel="records")
plt.tight_layout()
plt.show()

---

## What the EDA decided

1. **Target-group background sampling is viable and necessary.** The background
   pool covers the presence locations (density correlation is high), and the
   clustering visible in §1 is exactly the bias that needs to cancel. Random
   ocean background is rejected.
2. **Depth and distance-to-coast will dominate.** They separate the species far
   more cleanly than temperature. Expect this in the SHAP ranking, and interpret
   it with the sampling caveat from §4 in mind.
3. **Raw latitude/longitude must be excluded from the model.** Presences are
   spatially clustered enough that a tree model would memorise coordinates and
   score well under a naive split while learning no ecology. Spatial structure
   enters through environment and bathymetry instead.
4. **Nutrient concentrations are near-redundant.** Add ratios (N:P, N:Si) which
   carry limiting-nutrient information the raw values do not.
5. **The June–September sampling gap is an artefact.** Monsoon phase goes in as
   an explicit categorical state.
6. **Frontal and eddy signals are real but modest.** Include them; do not expect
   them to lead.

Next: **`02_feature_engineering`** builds the labelled training table —
including the pseudo-absence construction that decision 1 above commits us to.